# Chẩn đoán đường quan hệ → bằng chứng → decoder: F, 20 ảnh
**Không huấn luyện, không chọn checkpoint, không đánh giá hiệu năng benchmark.** Hai T4 chia nhau suy luận cùng F.
Bốn điều kiện: tắt nhánh / UPT / annotation dương / annotation âm. Hai điều kiện annotation giữ cùng box GT và ba slot S/O/R, chỉ đổi text quan hệ.
Nhãn âm V-COCO có thể vẫn mô tả hành động hợp lý; cần xem ảnh, không tự coi đây là đúng/sai tuyệt đối.
Bạn tự chạy Save & Run All, Internet ON, GPU T4 ×2. Không cần CHAIR, object detections hay huấn luyện lại.

In [ ]:
print('F RELATION CONDITIONING DIAGNOSTIC: NO TRAINING',flush=True)
from pathlib import Path
# Nếu chạy ngay sau F trong cùng session, đường dẫn này có sẵn.
# Nếu notebook mới: sửa thành đường dẫn checkpoint F trong Input, giữ đúng SHA4938bfe8...
CHECKPOINT=Path('/kaggle/working/semantic_evidence_verified_pairs_F/checkpoint.pth')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
UPT_PROMPT=Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
COCO_IMAGES=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
OUTPUT=Path('/kaggle/working/relation_conditioning_diagnostic_20')


In [ ]:
import json,os,subprocess,sys,hashlib,base64,zlib,shutil
os.environ.update(PYTHONUNBUFFERED='1',HDF5_USE_FILE_LOCKING='FALSE',CUBLAS_WORKSPACE_CONFIG=':4096:8',
    TOKENIZERS_PARALLELISM='false',PYTHONHASHSEED='42',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',HF_HUB_DISABLE_XET='1')
REPO=Path('/tmp/Image_Captioning-relation-conditioning')
LOG=Path('/tmp/relation_conditioning_setup.log')
def run(args,cwd=None):
    print('Running:',' '.join(map(str,args)),flush=True)
    with LOG.open('a',encoding='utf-8') as log:
        with subprocess.Popen(list(map(str,args)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1) as process:
            for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            status=process.wait()
    if status:raise RuntimeError(f'Command failed ({status}); log: {LOG}')
for path in (CHECKPOINT,LEGACY_PROMPT,UPT_PROMPT,COCO_JSON):assert path.is_file(),f'Copy actual mounted path: {path}'
assert COCO_IMAGES.is_dir() and VISUAL_CACHE.is_dir()
assert not OUTPUT.exists(),'Choose a fresh OUTPUT/restart session'
import torch
assert torch.cuda.device_count()==2,'Select T4 x2; one frozen checkpoint across both GPUs'
if not REPO.exists():run(['git','clone','--no-checkout','https://github.com/Supzxjee/Image_Captioning.git',REPO])
run(['git','fetch','--depth','1','origin','fa0087fe5b94b9b9db5bb0ba865683286e7e6e97'],cwd=REPO)
run(['git','checkout','--detach','fa0087fe5b94b9b9db5bb0ba865683286e7e6e97'],cwd=REPO)
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow','tqdm'])

PAYLOAD_B64 = '...'
PAYLOAD_SHA = 'b40aa2cc499123ef2d46dc1d450568692e979898cec97d74b36f953fbe4191ce'
raw=zlib.decompress(base64.b64decode(PAYLOAD_B64,validate=True))
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA
bundle=json.loads(raw)
SOURCES=bundle['sources']
SOURCE_HASHES = {'base_runner.py': 'b8a58ecdccf89151aedbedad3ca0ad73832438db8bae707cdcf1aa5800830801', 'pilot_model.py': '88cb1f4587b88d1400ab05ac7aa8f8f0535a9e16c698f60036270290df9a3d59', 'evidence_core.py': '4f8189de7d183bb062247043d6a2ce7bcb6fbd9fb7d0d485922c065d8cbe962b', 'pilot_data.py': 'aac539b7c23f4acc3ac05b99496ef8eb79f740e6c99d11742b6de08fd2b892c0', 'probe_data.py': 'f8dfb24cf825ecc2935bfc062731009197459cd55ebb9b0cbb146d248d574e9f', 'synthetic_fixtures.py': '44671927a64ed6d9d0c3a09137fcf9e680641ebba64e29d1d233945026ea1988', 'oracle_data.py': '885656552e24215c39ece453587b0ede7dc6d0f132b705fda8b513f30534baeb', 'review_html.py': 'f3948c9d9c73350cbcccd9e6722e7d5ba72d32332b6dab642abf767c85bf685d', 'run_diagnostic.py': '0156dbb6dcfa371f9567ff7d9c30a41058e962a7a708d03928c057546019c99e', 'test_oracle_data.py': '791af0fc4e7a3b7b7def65594bdaeac7ce06211ef20ed2d35bafa59d48407a88', 'test_oracle_model.py': '16d99aa934087b768a09df3b749faac5ebc0cf0204ba7fb21da644554d24ad31'}
for name,source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==SOURCE_HASHES[name]
    (REPO/name).write_text(source,encoding='utf-8')
MANIFEST=REPO/'cases_manifest.json'
MANIFEST.write_text(bundle['manifest'],encoding='utf-8')
for row in json.loads(bundle['manifest'])['cases']:
    assert any(p.is_file() for p in (COCO_IMAGES/row['filepath']/row['filename'],COCO_IMAGES/row['filename'])),f'Missing raw review image: {row["filename"]}; check COCO_IMAGES before GPU inference'
OUTPUT.mkdir(parents=True)
(OUTPUT/'source').mkdir()
for name,source in SOURCES.items():(OUTPUT/'source'/name).write_text(source,encoding='utf-8')
shutil.copy2(MANIFEST,OUTPUT/'cases_manifest.json')
cfg=dict(checkpoint=str(CHECKPOINT),visual_cache=str(VISUAL_CACHE),legacy_prompt=str(LEGACY_PROMPT),upt_prompt=str(UPT_PROMPT),dataset_json=str(COCO_JSON),coco_images=str(COCO_IMAGES),output=str(OUTPUT),manifest=str(MANIFEST),manifest_sha256=hashlib.sha256(bundle['manifest'].encode()).hexdigest(),source_sha256=SOURCE_HASHES,project_commit='fa0087fe5b94b9b9db5bb0ba865683286e7e6e97')
CONFIG=OUTPUT/'config.json'
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')


## Kiểm tra trước khi suy luận
Kiểm tra cách chọn ca, giữ box/slot cố định và module frozen bằng dữ liệu giả. Chưa thực hiện huấn luyện.

In [ ]:
for name in ('test_oracle_data.py','test_oracle_model.py'):run([sys.executable,'-u',REPO/name],cwd=REPO)
print('SYNTHETIC PREFLIGHT PASS',flush=True)
shutil.copy2(LOG,OUTPUT/'setup.log')
LOG=OUTPUT/'run.log'


## Suy luận bốn điều kiện và xuất review
Hai worker độc lập, mỗi GPU 10 ảnh; không optimizer/backward/DDP collectives.
Kiểm tra checkpoint/nhãn/cache bằng hash, replay điểm matcher F, xác nhận trọng số không đổi.
ZIP gồm 20 ảnh, box, caption, likelihood và HTML để kiểm tra nhãn. Không tự chấm caption đúng chỉ bằng từ khóa.

In [ ]:
try:
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',REPO/'run_diagnostic.py','worker','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u',REPO/'run_diagnostic.py','finish','--config',CONFIG],cwd=REPO)
finally:
    run([sys.executable,'-u',REPO/'run_diagnostic.py','archive','--config',CONFIG],cwd=REPO)
print('SEND:',str(OUTPUT)+'_review.zip',flush=True)
print('REVIEW:',OUTPUT/'case_review_20.html',flush=True)
